# OD & OC Segmentation from Retinal Fundus Photography

**Authors:** Dwi Rezky Fahlan · Sola Gracia Rania  
**Course:** EB3206 Biomedical Image Processing

This notebook implements a classical pipeline for Optic Disc (OD) and Optic Cup (OC) segmentation from fundus images using the Drishti-GS dataset.

Pipeline:
1. EDA → informs design decisions
2. OD Localization (sliding window)
3. OD Segmentation (Otsu on red channel)
4. OD Post-processing (morphological ops + ellipse Hough)
5. OC Pre-processing (histogram equalization)
6. OC Segmentation (K-means, multiple experiments)
7. OC Post-processing (morphological ops + ellipse Hough)

## Configuration

Set `BASE_DIR` to the root of your Drishti-GS dataset. All paths are derived from this.

In [ ]:
import os

# ─── SET THIS ────────────────────────────────────────────────────────────────
BASE_DIR = '/path/to/drishti-gs'  # e.g. '/content/drive/MyDrive/drishti-gs'
# ─────────────────────────────────────────────────────────────────────────────

# Training paths
TRAIN_IMAGES     = os.path.join(BASE_DIR, 'Training', 'Images_training')
TRAIN_OD_LABELS  = os.path.join(BASE_DIR, 'Training', 'OD_training')
TRAIN_OC_LABELS  = os.path.join(BASE_DIR, 'Training', 'OC_training')
TRAIN_SEG_OD     = os.path.join(BASE_DIR, 'Training', 'SegmentedOD')
TRAIN_LOCALIZED  = os.path.join(BASE_DIR, 'Training', 'LocalizedOD')
TRAIN_ELLIPSE_OC = os.path.join(BASE_DIR, 'Training', 'EllipseOC')
TRAIN_FINAL_OC   = os.path.join(BASE_DIR, 'Training', 'FinalOC')

# Testing paths
TEST_IMAGES      = os.path.join(BASE_DIR, 'Testing', 'Image_testing')
TEST_OD_LABELS   = os.path.join(BASE_DIR, 'Testing', 'OD_testing')
TEST_OC_LABELS   = os.path.join(BASE_DIR, 'Testing', 'OC_testing')
TEST_LOCALIZED   = os.path.join(BASE_DIR, 'Testing', 'LocalizedOD')
TEST_SEG_OD      = os.path.join(BASE_DIR, 'Testing', 'SegmentedOD')
TEST_OPENED_OD   = os.path.join(BASE_DIR, 'Testing', 'OpenedOD')
TEST_CLOSED_OD   = os.path.join(BASE_DIR, 'Testing', 'ClosedOD')
TEST_ELLIPSE_OD  = os.path.join(BASE_DIR, 'Testing', 'EllipseOD')
TEST_FINAL_OD    = os.path.join(BASE_DIR, 'Testing', 'FinalOD')
TEST_SEG_OC      = os.path.join(BASE_DIR, 'Testing', 'SegmentedOC')
TEST_CLOSED_OC   = os.path.join(BASE_DIR, 'Testing', 'ClosedOC')
TEST_OPENED_OC   = os.path.join(BASE_DIR, 'Testing', 'OpenedOC')
TEST_ELLIPSE_OC  = os.path.join(BASE_DIR, 'Testing', 'EllipseOC')
TEST_FINAL_OC    = os.path.join(BASE_DIR, 'Testing', 'FinalOC')

# Mask directories (all OD/OC masks combined)
OD_ALL = os.path.join(BASE_DIR, 'All', 'OD_all')
OC_ALL = os.path.join(BASE_DIR, 'All', 'OC_all')
IMG_ALL = os.path.join(BASE_DIR, 'All', 'Image_all')

print('Paths configured.')

## Imports

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw
from scipy.signal import convolve2d
from skimage import exposure
from skimage.morphology import disk, closing, opening
from skimage.io import imread, imsave
from skimage.color import rgb2gray
from sklearn.metrics import f1_score

---
# 1. Exploratory Data Analysis

Before building the pipeline, we analyze the dataset to understand OD and OC mask characteristics. The findings directly inform method choices downstream.

## 1.1 OD Mask Statistics

We examine OD mask size, bounding box dimensions, fill ratio, and spatial distribution of centers.

In [ ]:
def process_mask(image_path):
    """Extract bounding box stats from a binary mask image."""
    with Image.open(image_path) as img:
        img_array = np.array(img)
        white_pixels = np.argwhere(img_array == 255)
        if white_pixels.size == 0:
            return None

        min_y, min_x = white_pixels.min(axis=0)
        max_y, max_x = white_pixels.max(axis=0)
        length = max_y - min_y + 1
        width  = max_x - min_x + 1
        area   = length * width
        bbox   = img_array[min_y:max_y+1, min_x:max_x+1]
        white_pixel_count = np.sum(bbox == 255)
        ratio  = white_pixel_count / area
        center_x = min_x + width // 2
        center_y = min_y + length // 2

        return {
            'file': os.path.basename(image_path),
            'white_pixel_count': white_pixel_count,
            'box_area': area,
            'box_length': length,
            'box_width': width,
            'ratio': ratio,
            'box_center': (center_x, center_y),
            'bounding_box': (min_x, min_y, max_x, max_y)
        }

In [ ]:
# Process all OD masks
dataOD = []
for f in os.listdir(OD_ALL):
    path = os.path.join(OD_ALL, f)
    if os.path.isfile(path):
        result = process_mask(path)
        if result:
            dataOD.append(result)

# Summary statistics
white_counts = [d['white_pixel_count'] for d in dataOD]
print(f"OD Mask Statistics")
print(f"  Mean white area : {np.mean(white_counts):.1f}")
print(f"  Std             : {np.std(white_counts):.1f}")
print(f"  Min             : {np.min(white_counts)}")
print(f"  Max             : {np.max(white_counts)}")

In [ ]:
# Scatter plot of OD bounding box centers
# Finding: centers cluster tightly, confirming sliding window localization is viable
center_points = [d['box_center'] for d in dataOD if d['box_center'] != (0, 0)]
files = os.listdir(OD_ALL)
first_image_path = os.path.join(OD_ALL, files[0])
with Image.open(first_image_path) as img:
    img_width, img_height = img.size

x_coords, y_coords = zip(*center_points)
plt.figure(figsize=(10, 6))
plt.scatter(x_coords, y_coords, c='blue', marker='o')
plt.xlabel('Center X Coordinate')
plt.ylabel('Center Y Coordinate')
plt.title('Scatter Plot of OD Bounding Box Centers')
plt.xlim(0, img_width)
plt.ylim(0, img_height)
plt.gca().invert_yaxis()
plt.grid(True)
plt.show()

# Design decision: OD centers are spatially consistent across images.
# A sliding window approach is viable. Window radius of 330px is derived
# from the mean bounding box dimensions (~383x337px).

## 1.2 OC Mask Statistics

In [ ]:
dataOC = []
for f in os.listdir(OC_ALL):
    path = os.path.join(OC_ALL, f)
    if os.path.isfile(path):
        result = process_mask(path)
        if result:
            dataOC.append(result)

white_counts_oc = [d['white_pixel_count'] for d in dataOC]
print(f"OC Mask Statistics")
print(f"  Mean white area : {np.mean(white_counts_oc):.1f}")
print(f"  Std             : {np.std(white_counts_oc):.1f}")
print(f"  Min             : {np.min(white_counts_oc)}")
print(f"  Max             : {np.max(white_counts_oc)}")

## 1.3 OC/OD Size Ratio

The OC/OD white pixel ratio informs the expected number of intensity clusters in the OD region, which guides the choice of K in K-means.

In [ ]:
def count_white_pixels(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    return np.sum(img == 255)

oc_files = os.listdir(OC_ALL)
mask_ratios = []

for oc_file in oc_files:
    oc_number = ''.join(filter(str.isdigit, oc_file))
    od_file = f'Copy of drishtiGS_{oc_number}_ODAvgBoundary_OD_img.png'
    od_path = os.path.join(OD_ALL, od_file)
    oc_path = os.path.join(OC_ALL, oc_file)

    if os.path.exists(od_path):
        oc_px = count_white_pixels(oc_path)
        od_px = count_white_pixels(od_path)
        if od_px > 0:
            mask_ratios.append(oc_px / od_px)

print(f"OC/OD White Pixel Ratio")
print(f"  Mean : {np.mean(mask_ratios):.4f}")
print(f"  Std  : {np.std(mask_ratios):.4f}")
print(f"  Min  : {np.min(mask_ratios):.4f}")
print(f"  Max  : {np.max(mask_ratios):.4f}")

# Design decision: OC covers a distinct sub-region of OD. Combined with the
# visible intensity gradient (bright OC center, darker OD periphery, retinal
# background), this suggests K=3-4 as a reasonable starting range for K-means.

## 1.4 Channel Analysis

We inspect per-channel contrast to decide which channel to use for OD and OC segmentation.

In [ ]:
def display_image_channels(image_path):
    image = cv2.imread(image_path)
    if image is None:
        print(f"Failed to load: {image_path}")
        return
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    r = image_rgb[:, :, 0]
    g = image_rgb[:, :, 1]
    b = image_rgb[:, :, 2]
    grey = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    fig, axs = plt.subplots(1, 5, figsize=(20, 5))
    fig.suptitle(f'Channel Visualization: {os.path.basename(image_path)}')
    for ax, img, title, cmap in zip(
        axs,
        [image_rgb, r, g, b, grey],
        ['Original (RGB)', 'Red', 'Green', 'Blue', 'Greyscale'],
        [None, 'Reds', 'Greens', 'Blues', 'gray']
    ):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    plt.show()

image_files = [f for f in os.listdir(IMG_ALL) if f.lower().endswith(('png', 'jpg', 'jpeg'))][:2]
for f in image_files:
    display_image_channels(os.path.join(IMG_ALL, f))

# Observation: The red channel provides the clearest contrast between the
# bright OD region and surrounding retinal tissue. This justifies using
# the red channel as input for both OD (Otsu) and OC (K-means) segmentation.

## 1.5 Preprocessing Chain: Original → Cropped → Equalized

Histogram equalization on the localized OD region spreads the intensity distribution, improving separability between OC and the surrounding OD tissue.

In [ ]:
def show_preprocessing_chain(retina_path, segmented_od_path):
    retina = cv2.imread(retina_path)
    seg = cv2.imread(segmented_od_path)
    if retina is None or seg is None:
        return

    seg_gray = cv2.cvtColor(seg, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(seg_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return
    contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(contour)
    cropped = retina[y:y+h, x:x+w]

    r, g, b = cropped[:,:,2], cropped[:,:,1], cropped[:,:,0]
    r_eq = cv2.equalizeHist(r)
    g_eq = cv2.equalizeHist(g)
    b_eq = cv2.equalizeHist(b)
    equalized = cv2.merge((b_eq, g_eq, r_eq))

    # Image chain
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'Preprocessing Chain: {os.path.basename(retina_path)}')
    for ax, img, title in zip(axs,
        [cv2.cvtColor(retina, cv2.COLOR_BGR2RGB),
         cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB),
         cv2.cvtColor(equalized, cv2.COLOR_BGR2RGB)],
        ['Original', 'Cropped (Localized OD)', 'Equalized']):
        ax.imshow(img); ax.set_title(title); ax.axis('off')
    plt.show()

    # Histograms
    fig, axs = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('Channel Histograms Before/After Equalization')
    for i, (ch, ch_eq, color, name) in enumerate(zip(
        [r, g, b], [r_eq, g_eq, b_eq], ['r','g','b'], ['Red','Green','Blue']
    )):
        axs[0, i].plot(cv2.calcHist([ch], [0], None, [256], [0,256]), color=color)
        axs[0, i].set_title(f'{name} (Original)')
        axs[1, i].plot(cv2.calcHist([ch_eq], [0], None, [256], [0,256]), color=color)
        axs[1, i].set_title(f'{name} (Equalized)')
    plt.tight_layout()
    plt.show()

retina_files = sorted([f for f in os.listdir(TRAIN_IMAGES) if f.lower().endswith(('png','jpg','jpeg'))])
for f in retina_files[:2]:
    seg_path = os.path.join(TRAIN_SEG_OD, f'segmented_{f}')
    if os.path.exists(seg_path):
        show_preprocessing_chain(os.path.join(TRAIN_IMAGES, f), seg_path)

---
# 2. OD Localization

**Method:** Sliding circular window (radius=330px) over grayscale image; window position with maximum intensity sum is selected as OD location.

**Rationale:** EDA showed OD is the brightest region in fundus images and its center is spatially consistent across samples. The window radius of 330px is derived from mean OD bounding box dimensions (~383×337px).

In [ ]:
def create_circular_mask(window_size):
    radius = window_size // 2
    y, x = np.ogrid[:window_size, :window_size]
    center = (radius, radius)
    mask = (x - center[0])**2 + (y - center[1])**2 <= radius**2
    return mask

def find_max_intensity_window(image, window_size):
    max_intensity = -1
    max_x = max_y = None
    circular_mask = create_circular_mask(window_size)
    for y in range(image.shape[0] - window_size + 1):
        for x in range(image.shape[1] - window_size + 1):
            window = image[y:y+window_size, x:x+window_size]
            intensity = np.sum(window[circular_mask])
            if intensity > max_intensity:
                max_intensity = intensity
                max_x, max_y = x, y
    return max_x, max_y

def localize_image(image_path, window_size, output_path):
    image = Image.open(image_path).convert('RGB')
    image_array = np.array(image)
    grayscale = np.array(image.convert('L'))

    # Black out outer 12% to avoid artifacts at image borders
    h, w = grayscale.shape
    my, mx = int(h * 0.12), int(w * 0.12)
    grayscale[:my, :] = grayscale[-my:, :] = 0
    grayscale[:, :mx] = grayscale[:, -mx:] = 0

    max_x, max_y = find_max_intensity_window(grayscale, window_size)
    margin = 110
    start_y = max(0, max_y - margin)
    end_y   = min(image_array.shape[0], max_y + window_size + margin)
    start_x = max(0, max_x - margin)
    end_x   = min(image_array.shape[1], max_x + window_size + margin)

    localized = image_array[start_y:end_y, start_x:end_x]
    Image.fromarray(localized).save(output_path)
    print(f'Saved: {output_path}')

def process_localization_folder(input_folder, output_folder, window_size=330):
    os.makedirs(output_folder, exist_ok=True)
    for f in os.listdir(input_folder):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            localize_image(
                os.path.join(input_folder, f),
                window_size,
                os.path.join(output_folder, f'localized_{f}')
            )

process_localization_folder(TRAIN_IMAGES, TRAIN_LOCALIZED)

---
# 3. OD Segmentation

**Method:** Otsu thresholding on the red channel of the localized ROI.

**Rationale:** Channel analysis (EDA Section 1.4) showed the red channel provides the best contrast between OD and surrounding retinal tissue. Otsu thresholding automatically finds the optimal threshold without manual tuning.

In [ ]:
def segment_od(image_path, window_size, seg_output_path, loc_output_path):
    image = Image.open(image_path).convert('RGB')
    image_array = np.array(image)
    grayscale = np.array(image.convert('L'))

    h, w = grayscale.shape
    my, mx = int(h * 0.12), int(w * 0.12)
    grayscale[:my, :] = grayscale[-my:, :] = 0
    grayscale[:, :mx] = grayscale[:, -mx:] = 0

    max_x, max_y = find_max_intensity_window(grayscale, window_size)
    margin = 110
    start_y = max(0, max_y - margin)
    end_y   = min(image_array.shape[0], max_y + window_size + margin)
    start_x = max(0, max_x - margin)
    end_x   = min(image_array.shape[1], max_x + window_size + margin)

    localized = image_array[start_y:end_y, start_x:end_x]
    Image.fromarray(localized).save(loc_output_path)

    red_channel = localized[:, :, 0]
    _, thresholded = cv2.threshold(red_channel, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    full_seg = np.zeros_like(grayscale)
    full_seg[start_y:end_y, start_x:end_x] = thresholded
    cv2.imwrite(seg_output_path, full_seg)
    print(f'Saved: {seg_output_path}')

def process_od_segmentation(input_folder, window_size, seg_folder, loc_folder):
    os.makedirs(seg_folder, exist_ok=True)
    os.makedirs(loc_folder, exist_ok=True)
    for f in os.listdir(input_folder):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            segment_od(
                os.path.join(input_folder, f),
                window_size,
                os.path.join(seg_folder, f'segmented_{f}'),
                os.path.join(loc_folder, f'localized_{f}')
            )

process_od_segmentation(TEST_IMAGES, 330, TEST_SEG_OD, TEST_LOCALIZED)

In [ ]:
# F1 Evaluation — OD Segmentation
def compute_od_f1(segmented_folder, label_folder):
    f1_scores = []
    for seg_file in os.listdir(segmented_folder):
        if not seg_file.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        common = seg_file.replace('segmented_', '').replace('.png', '')
        label_file = f"{common}_ODAvgBoundary_OD_img.png"
        label_path = os.path.join(label_folder, label_file)
        if not os.path.exists(label_path):
            continue
        seg_img   = cv2.imread(os.path.join(segmented_folder, seg_file), cv2.IMREAD_GRAYSCALE)
        label_img = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)
        _, seg_bin   = cv2.threshold(seg_img, 127, 1, cv2.THRESH_BINARY)
        _, label_bin = cv2.threshold(label_img, 127, 1, cv2.THRESH_BINARY)
        f1 = f1_score(label_bin.flatten(), seg_bin.flatten(), average='binary')
        f1_scores.append((seg_file, f1))

    scores = [s for _, s in f1_scores]
    print(f"OD Segmentation F1 — Mean: {np.mean(scores):.4f}, Min: {np.min(scores):.4f}, Max: {np.max(scores):.4f}")
    return f1_scores

compute_od_f1(TEST_SEG_OD, TEST_OD_LABELS)

---
# 4. OD Post-Processing

**Method:** Morphological opening → closing (disk radius=30) → Ellipse Hough Transform.

Opening removes small noise regions; closing fills gaps in the OD boundary. Ellipse Hough Transform regularizes the result into a smooth elliptical shape consistent with OD anatomy. Disk radius was determined empirically.

In [ ]:
def morphological_op(input_folder, output_folder, operation='opening', radius=30):
    os.makedirs(output_folder, exist_ok=True)
    selem = disk(radius)
    op_fn = opening if operation == 'opening' else closing
    for f in os.listdir(input_folder):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            img = imread(os.path.join(input_folder, f), as_gray=True)
            result = op_fn(img, selem)
            imsave(os.path.join(output_folder, f), result)
    print(f'{operation.capitalize()} complete → {output_folder}')

morphological_op(TEST_SEG_OD, TEST_OPENED_OD, operation='opening')
morphological_op(TEST_OPENED_OD, TEST_CLOSED_OD, operation='closing')

In [ ]:
def apply_ellipse_hough(input_folder, output_folder, scale_down=1.0):
    os.makedirs(output_folder, exist_ok=True)
    for f in os.listdir(input_folder):
        if not f.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        image = imread(os.path.join(input_folder, f))
        if len(image.shape) == 3:
            image_gray = rgb2gray(image)
        else:
            image_gray = image
        image_uint8 = (image_gray * 255).astype(np.uint8)
        contours, _ = cv2.findContours(image_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        ellipse_mask = np.zeros(image_gray.shape, dtype=np.uint8)
        for contour in contours:
            if len(contour) >= 5:
                ellipse = cv2.fitEllipse(contour)
                if scale_down != 1.0:
                    center, axes, angle = ellipse
                    axes = (axes[0] * scale_down, axes[1] * scale_down)
                    ellipse = (center, axes, angle)
                cv2.ellipse(ellipse_mask, ellipse, 255, thickness=-1)
        imsave(os.path.join(output_folder, f), ellipse_mask)
    print(f'Ellipse Hough complete → {output_folder}')

apply_ellipse_hough(TEST_CLOSED_OD, TEST_ELLIPSE_OD)

In [ ]:
# F1 Evaluation — OD Post-Processing
def compute_od_ellipse_f1(ellipse_folder, label_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    f1_scores = []
    for ef in os.listdir(ellipse_folder):
        if not ef.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        image_number = ef.split('_')[2].split('.')[0]
        label_file = f"drishtiGS_{image_number}_ODAvgBoundary_OD_img.png"
        label_path = os.path.join(label_folder, label_file)
        if not os.path.exists(label_path):
            continue

        ellipse_img = imread(os.path.join(ellipse_folder, ef), as_gray=True)
        label_img   = imread(label_path, as_gray=True)
        e_bin = ellipse_img > 0
        l_bin = label_img > 0

        tp = np.logical_and(e_bin, l_bin)
        tn = np.logical_and(~e_bin, ~l_bin)
        fp = np.logical_and(e_bin, ~l_bin)
        fn = np.logical_and(~e_bin, l_bin)

        annotated = np.zeros((*ellipse_img.shape, 3), dtype=np.uint8)
        annotated[tn] = [0, 0, 0]
        annotated[tp] = [0, 255, 0]
        annotated[fn] = [255, 0, 0]
        annotated[fp] = [255, 255, 0]
        imsave(os.path.join(output_folder, ef), annotated)

        f1 = f1_score(l_bin.flatten(), e_bin.flatten())
        f1_scores.append((ef, f1))

    scores = [s for _, s in f1_scores]
    print(f"OD Ellipse F1 — Mean: {np.mean(scores):.4f}, Min: {np.min(scores):.4f}, Max: {np.max(scores):.4f}")
    return pd.DataFrame(f1_scores, columns=['Filename', 'F1 Score'])

df_od = compute_od_ellipse_f1(TEST_ELLIPSE_OD, TEST_OD_LABELS, TEST_FINAL_OD)
print(df_od)

---
# 5. OC Segmentation Experiments

We compare four variants to find the best configuration:
- K-means K=3 on red channel
- K-means K=4 on red channel  ← **best result**
- K-means K=3 on equalized red channel
- K-means K=4 on equalized red channel

K range was estimated from OC/OD size ratio analysis (EDA Section 1.3). Final K was selected empirically based on F1 scores.

In [ ]:
def segment_oc_kmeans(retina_path, seg_od_path, oc_label_path, k=4, equalize=False):
    """Segment OC using K-means on the red channel of the localized OD region."""
    retina  = cv2.imread(retina_path)
    seg_od  = cv2.imread(seg_od_path)
    oc_label = cv2.imread(oc_label_path, cv2.IMREAD_GRAYSCALE)
    if retina is None or seg_od is None or oc_label is None:
        return None

    seg_gray = cv2.cvtColor(seg_od, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(seg_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(contour)

    cropped = retina[y:y+h, x:x+w]
    r_channel = cropped[:, :, 2]
    if equalize:
        r_channel = cv2.equalizeHist(r_channel)

    pixel_values = np.float32(r_channel.reshape((-1, 1)))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 1000, 0.05)
    _, labels, centers = cv2.kmeans(pixel_values, k, None, criteria, 5, cv2.KMEANS_RANDOM_CENTERS)
    centers = np.uint8(centers)

    oc_cluster = np.argmax(centers)
    oc_mask = (labels.flatten() == oc_cluster).astype(np.uint8).reshape(r_channel.shape) * 255

    full_mask = np.zeros(retina.shape[:2], dtype=np.uint8)
    resized = cv2.resize(oc_mask, (w, h), interpolation=cv2.INTER_NEAREST)
    full_mask[y:y+h, x:x+w] = resized

    selem = disk(30)
    closed = closing(full_mask, selem)

    true_labels = oc_label.flatten() // 255
    pred_labels = closed.flatten() // 255
    f1_before = f1_score(true_labels, pred_labels, average='binary')

    contours2, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    results = {'f1_before_ellipse': f1_before}
    for scale, label in [(1.0, 'no_scale'), (0.8, 'scale_20'), (0.9, 'scale_10')]:
        em = np.zeros_like(closed)
        for c in contours2:
            if len(c) >= 5:
                e = cv2.fitEllipse(c)
                if scale != 1.0:
                    center, axes, angle = e
                    axes = (axes[0]*scale, axes[1]*scale)
                    e = (center, axes, angle)
                cv2.ellipse(em, e, 255, thickness=-1)
        f1 = f1_score(true_labels, em.flatten()//255, average='binary')
        results[f'f1_{label}'] = f1

    return results

def run_oc_experiment(retina_folder, seg_od_folder, oc_label_folder, k, equalize, label):
    retina_files = [f for f in os.listdir(retina_folder) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    all_results = []
    for f in retina_files:
        base = os.path.splitext(f)[0]
        seg_path   = os.path.join(seg_od_folder, f'segmented_{base}.png')
        label_path = os.path.join(oc_label_folder, f'{base}_CupAvgBoundary_OC_img.png')
        if not os.path.exists(seg_path) or not os.path.exists(label_path):
            continue
        r = segment_oc_kmeans(os.path.join(retina_folder, f), seg_path, label_path, k=k, equalize=equalize)
        if r:
            all_results.append({'file': f, **r})

    df = pd.DataFrame(all_results)
    print(f"\n--- {label} ---")
    for col in [c for c in df.columns if 'f1' in c]:
        print(f"  {col}: mean={df[col].mean():.4f}, min={df[col].min():.4f}, max={df[col].max():.4f}")
    return df

In [ ]:
# Experiment 1: K=3, red channel
df_k3 = run_oc_experiment(TRAIN_IMAGES, TRAIN_SEG_OD, TRAIN_OC_LABELS, k=3, equalize=False, label='K=3, Red Channel')

In [ ]:
# Experiment 2: K=4, red channel
df_k4 = run_oc_experiment(TRAIN_IMAGES, TRAIN_SEG_OD, TRAIN_OC_LABELS, k=4, equalize=False, label='K=4, Red Channel')

In [ ]:
# Experiment 3: K=3, equalized red channel
df_k3_eq = run_oc_experiment(TRAIN_IMAGES, TRAIN_SEG_OD, TRAIN_OC_LABELS, k=3, equalize=True, label='K=3, Equalized Red')

In [ ]:
# Experiment 4: K=4, equalized red channel
df_k4_eq = run_oc_experiment(TRAIN_IMAGES, TRAIN_SEG_OD, TRAIN_OC_LABELS, k=4, equalize=True, label='K=4, Equalized Red')

---
# 6. OC Final Segmentation (Best Method: K=4, Red Channel)

Based on the experiments above, K=4 on the raw red channel achieved the best F1. Applied here to the test set.

In [ ]:
def segment_oc_final(retina_path, seg_od_path, output_folder, k=4):
    retina = cv2.imread(retina_path)
    seg_od = cv2.imread(seg_od_path)
    if retina is None or seg_od is None:
        return None

    seg_gray = cv2.cvtColor(seg_od, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(seg_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(contour)
    cropped = retina[y:y+h, x:x+w]

    r_channel = cropped[:, :, 2]
    pixel_values = np.float32(r_channel.reshape((-1, 1)))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 1000, 0.05)
    _, labels, centers = cv2.kmeans(pixel_values, k, None, criteria, 5, cv2.KMEANS_RANDOM_CENTERS)
    centers = np.uint8(centers)

    oc_cluster = np.argmax(centers)
    oc_mask = (labels.flatten() == oc_cluster).astype(np.uint8).reshape(r_channel.shape) * 255
    full_mask = np.zeros(retina.shape[:2], dtype=np.uint8)
    full_mask[y:y+h, x:x+w] = cv2.resize(oc_mask, (w, h), interpolation=cv2.INTER_NEAREST)

    output_path = os.path.join(output_folder, os.path.basename(retina_path))
    imsave(output_path, full_mask)
    return full_mask

os.makedirs(TEST_SEG_OC, exist_ok=True)
for f in os.listdir(TEST_IMAGES):
    if f.lower().endswith(('.png','.jpg','.jpeg')):
        base = os.path.splitext(f)[0]
        seg_path = os.path.join(TEST_ELLIPSE_OD, f'{base}.png')
        if os.path.exists(seg_path):
            segment_oc_final(os.path.join(TEST_IMAGES, f), seg_path, TEST_SEG_OC)

---
# 7. OC Post-Processing

**Method:** Morphological closing → opening (disk radius=30) → Ellipse Hough Transform.

Same post-processing strategy as OD. Disk radius determined empirically.

In [ ]:
morphological_op(TEST_SEG_OC, TEST_CLOSED_OC, operation='closing')
morphological_op(TEST_CLOSED_OC, TEST_OPENED_OC, operation='opening')
apply_ellipse_hough(TEST_OPENED_OC, TEST_ELLIPSE_OC)

In [ ]:
# F1 Evaluation — OC Post-Processing
def compute_oc_ellipse_f1(ellipse_folder, label_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    f1_scores = []
    for ef in os.listdir(ellipse_folder):
        if not ef.lower().endswith(('.png','.jpg','.jpeg')):
            continue
        base = os.path.splitext(ef)[0]
        label_path = os.path.join(label_folder, f'{base}_CupAvgBoundary_OC_img.png')
        if not os.path.exists(label_path):
            continue

        ellipse_img = imread(os.path.join(ellipse_folder, ef), as_gray=True)
        label_img   = imread(label_path, as_gray=True)
        e_bin = ellipse_img > 0
        l_bin = label_img > 0

        tp = np.logical_and(e_bin, l_bin)
        tn = np.logical_and(~e_bin, ~l_bin)
        fp = np.logical_and(e_bin, ~l_bin)
        fn = np.logical_and(~e_bin, l_bin)

        annotated = np.zeros((*ellipse_img.shape, 3), dtype=np.uint8)
        annotated[tn] = [0, 0, 0]
        annotated[tp] = [0, 255, 0]
        annotated[fn] = [255, 0, 0]
        annotated[fp] = [255, 255, 0]
        imsave(os.path.join(output_folder, ef), annotated)

        f1 = f1_score(l_bin.flatten(), e_bin.flatten())
        f1_scores.append((ef, f1))

    scores = [s for _, s in f1_scores]
    print(f"OC Ellipse F1 — Mean: {np.mean(scores):.4f}, Min: {np.min(scores):.4f}, Max: {np.max(scores):.4f}")
    return pd.DataFrame(f1_scores, columns=['Filename', 'F1 Score'])

df_oc = compute_oc_ellipse_f1(TEST_ELLIPSE_OC, TEST_OC_LABELS, TEST_FINAL_OC)
print(df_oc)